# Notebook 02B — Multi-Noise Decoder Training

Retrains the **MLP** and **LSTM** decoders on the mixed-noise dataset from notebook 01B.  
Everything else is identical to notebooks 02 and 05 — same architectures, same hyperparameters.  
The only change is the training data.

**Expected improvement:** Both decoders should remain competitive across all noise levels,  
instead of collapsing to trivial performance at p ≥ 0.01.

**Prerequisite:** Run notebook 01B first.
```bash
pip install torch scikit-learn numpy
```

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import time, os, warnings
warnings.filterwarnings('ignore')

SEED   = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")

Device  : cuda
PyTorch : 2.10.0+cu128


---
## 1. Load Mixed-Noise Data

In [2]:
det = np.load("data/mixed/detection_events.npy").astype(np.float32)
raw = np.load("data/mixed/raw_measurements.npy").astype(np.float32)
obs = np.load("data/mixed/observable_flips.npy").astype(np.float32).squeeze()

print("Mixed-noise training data:")
print(f"  det : {det.shape}")
print(f"  raw : {raw.shape}")
print(f"  obs : {obs.shape}")
print(f"  Overall LER : {obs.mean():.4f}")
print()
print("This dataset contains equal shots from p=0.0005 through p=0.02.")
print("The model will see the full noise range during training.")

Mixed-noise training data:
  det : (99996, 72)
  raw : (99996, 72)
  obs : (99996,)
  Overall LER : 0.2009

This dataset contains equal shots from p=0.0005 through p=0.02.
The model will see the full noise range during training.


---
## 2. Train / Val / Test Split

In [3]:
idx = np.arange(len(obs))
idx_tr, idx_tmp = train_test_split(idx, test_size=0.30,
                                    stratify=obs.astype(int), random_state=SEED)
idx_val, idx_te = train_test_split(idx_tmp, test_size=0.50,
                                    stratify=obs[idx_tmp].astype(int), random_state=SEED)

det_tr,  det_val,  det_te  = det[idx_tr],  det[idx_val],  det[idx_te]
raw_tr,  raw_val,  raw_te  = raw[idx_tr],  raw[idx_val],  raw[idx_te]
obs_tr,  obs_val,  obs_te  = obs[idx_tr],  obs[idx_val],  obs[idx_te]

trivial_ler = obs_te.mean()
print(f"Split → train: {len(idx_tr):,}  val: {len(idx_val):,}  test: {len(idx_te):,}")
print(f"Trivial LER (mixed noise) : {trivial_ler:.4f}")
print()
print("Note: trivial LER is now the average over all noise levels (~20-25%)")

Split → train: 69,997  val: 14,999  test: 15,000
Trivial LER (mixed noise) : 0.2009

Note: trivial LER is now the average over all noise levels (~20-25%)


---
## 3. Model Definitions & Training Infrastructure

Identical architectures to notebooks 02 and 05.

In [4]:
# ── MLP ───────────────────────────────────────────────────────────────────────
class SurfaceCodeDecoder(nn.Module):
    def __init__(self, input_dim, hidden=[256, 128, 64], dropout=0.3):
        super().__init__()
        layers, in_d = [], input_dim
        for i, h in enumerate(hidden):
            layers += [nn.Linear(in_d, h), nn.BatchNorm1d(h), nn.ReLU(),
                       nn.Dropout(dropout if i < len(hidden)-1 else 0.0)]
            in_d = h
        layers.append(nn.Linear(in_d, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(-1)


# ── LSTM ──────────────────────────────────────────────────────────────────────
class LSTMDecoder(nn.Module):
    def __init__(self, input_size=8, lstm_hidden=36, lstm_layers=2,
                 dense=[48, 24, 12], dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, lstm_hidden, lstm_layers,
                            batch_first=True,
                            dropout=dropout if lstm_layers > 1 else 0.0)
        layers, in_d = [], lstm_hidden
        for h in dense:
            layers += [nn.Linear(in_d, h), nn.ReLU(), nn.Dropout(dropout)]
            in_d = h
        layers.append(nn.Linear(in_d, 1))
        self.head = nn.Sequential(*layers)
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.head(h_n[-1]).squeeze(-1)


# ── Shared utilities ──────────────────────────────────────────────────────────
def make_loader(X, y, batch_size=512, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

@torch.no_grad()
def predict(model, X, batch_size=1024):
    model.eval()
    ds = TensorDataset(torch.tensor(X))
    loader = DataLoader(ds, batch_size=batch_size)
    out = []
    for (Xb,) in loader:
        out.append((torch.sigmoid(model(Xb.to(DEVICE))) > 0.5).cpu().numpy())
    return np.concatenate(out)


def train_model(model, X_tr, y_tr, X_val, y_val,
                label="", epochs=120, lr=1e-3, patience=15):
    opt   = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sch   = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=6, factor=0.5)
    pw    = torch.tensor([(1-y_tr.mean())/y_tr.mean()]).to(DEVICE)
    crit  = nn.BCEWithLogitsLoss(pos_weight=pw)
    tr_l  = make_loader(X_tr,  y_tr)
    va_l  = make_loader(X_val, y_val, shuffle=False)

    hist = {'tr_ler': [], 'val_ler': [], 'tr_loss': [], 'val_loss': []}
    best_vloss, best_ep, best_state = np.inf, 0, None
    t0 = time.time()

    for ep in range(1, epochs+1):
        model.train()
        tl, tc, tt = 0.0, 0, 0
        for Xb, yb in tr_l:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(Xb)
            loss   = crit(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tc += ((torch.sigmoid(logits) > 0.5).float() == yb).sum().item()
            tt += len(yb); tl += loss.item()*len(yb)
        tr_ler = 1 - tc/tt; tr_loss = tl/tt

        model.eval()
        vl, vc, vt = 0.0, 0, 0
        with torch.no_grad():
            for Xb, yb in va_l:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                logits = model(Xb)
                vl += crit(logits, yb).item()*len(yb)
                vc += ((torch.sigmoid(logits)>0.5).float()==yb).sum().item()
                vt += len(yb)
        val_ler = 1-vc/vt; val_loss = vl/vt

        sch.step(val_loss)
        for k, v in zip(['tr_loss','val_loss','tr_ler','val_ler'],
                        [tr_loss, val_loss, tr_ler, val_ler]):
            hist[k].append(v)

        if val_loss < best_vloss:
            best_vloss = val_loss; best_ep = ep
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if ep % 20 == 0 or ep == 1:
            print(f"  [{label}] Epoch {ep:3d}  tr_loss={tr_loss:.4f}  "
                  f"val_loss={val_loss:.4f}  val_LER={val_ler:.5f}")

        if ep - best_ep >= patience:
            print(f"  [{label}] Early stop at epoch {ep} (best={best_ep})")
            break

    model.load_state_dict(best_state)
    return model, hist, best_ep


ROUNDS, STABS = 9, 8
print("Architectures defined.")
print(f"  MLP  : {sum(p.numel() for p in SurfaceCodeDecoder(72).parameters()):,} params")
print(f"  LSTM : {sum(p.numel() for p in LSTMDecoder().parameters()):,} params")

Architectures defined.
  MLP  : 60,801 params
  LSTM : 20,545 params


---
## 4. Train MLP — Detection Events (Part 1)

In [5]:
print("=" * 58)
print("MLP — Detection Events (Part 1) — Multi-noise")
print("=" * 58)

mlp_p1 = SurfaceCodeDecoder(72).to(DEVICE)
mlp_p1, hist_mlp_p1, best_ep = train_model(
    mlp_p1, det_tr, obs_tr, det_val, obs_val, label="MLP-P1")

pred_mlp_p1 = predict(mlp_p1, det_te)
ler_mlp_p1  = float((pred_mlp_p1 != obs_te).mean())
print(f"\n  Test LER : {ler_mlp_p1:.5f}  ({100*ler_mlp_p1:.4f}%)")
print(f"  Trivial  : {trivial_ler:.5f}  ({100*trivial_ler:.4f}%)")
print(f"  vs Trivial : {trivial_ler/ler_mlp_p1:.1f}x")

MLP — Detection Events (Part 1) — Multi-noise
  [MLP-P1] Epoch   1  tr_loss=0.7370  val_loss=0.6337  val_LER=0.27995
  [MLP-P1] Epoch  20  tr_loss=0.4123  val_loss=0.4672  val_LER=0.15181
  [MLP-P1] Epoch  40  tr_loss=0.3494  val_loss=0.4901  val_LER=0.13841
  [MLP-P1] Early stop at epoch 41 (best=26)

  Test LER : 0.15060  (15.0600%)
  Trivial  : 0.20093  (20.0933%)
  vs Trivial : 1.3x


---
## 5. Train MLP — Raw Syndromes (Part 2)

In [6]:
print("=" * 58)
print("MLP — Raw Syndromes (Part 2) — Multi-noise")
print("=" * 58)

mlp_p2 = SurfaceCodeDecoder(72).to(DEVICE)
mlp_p2, hist_mlp_p2, best_ep = train_model(
    mlp_p2, raw_tr, obs_tr, raw_val, obs_val, label="MLP-P2")

pred_mlp_p2 = predict(mlp_p2, raw_te)
ler_mlp_p2  = float((pred_mlp_p2 != obs_te).mean())
print(f"\n  Test LER : {ler_mlp_p2:.5f}  ({100*ler_mlp_p2:.4f}%)")
print(f"  Trivial  : {trivial_ler:.5f}  ({100*trivial_ler:.4f}%)")
print(f"  vs Trivial : {trivial_ler/ler_mlp_p2:.1f}x")

MLP — Raw Syndromes (Part 2) — Multi-noise
  [MLP-P2] Epoch   1  tr_loss=0.7080  val_loss=0.6372  val_LER=0.23928
  [MLP-P2] Epoch  20  tr_loss=0.5197  val_loss=0.5708  val_LER=0.17454
  [MLP-P2] Early stop at epoch 30 (best=15)

  Test LER : 0.17567  (17.5667%)
  Trivial  : 0.20093  (20.0933%)
  vs Trivial : 1.1x


---
## 6. Train LSTM — Detection Events

In [7]:
print("=" * 58)
print("LSTM — Detection Events — Multi-noise")
print("=" * 58)

det_tr_seq  = det_tr.reshape(-1, ROUNDS, STABS)
det_val_seq = det_val.reshape(-1, ROUNDS, STABS)
det_te_seq  = det_te.reshape(-1, ROUNDS, STABS)

lstm_mn = LSTMDecoder(STABS).to(DEVICE)
lstm_mn, hist_lstm_mn, best_ep = train_model(
    lstm_mn, det_tr_seq, obs_tr, det_val_seq, obs_val, label="LSTM")

pred_lstm_mn = predict(lstm_mn, det_te_seq)
ler_lstm_mn  = float((pred_lstm_mn != obs_te).mean())
print(f"\n  Test LER : {ler_lstm_mn:.5f}  ({100*ler_lstm_mn:.4f}%)")
print(f"  Trivial  : {trivial_ler:.5f}  ({100*trivial_ler:.4f}%)")
print(f"  vs Trivial : {trivial_ler/ler_lstm_mn:.1f}x")

LSTM — Detection Events — Multi-noise
  [LSTM] Epoch   1  tr_loss=0.9405  val_loss=0.7685  val_LER=0.33289
  [LSTM] Epoch  20  tr_loss=0.5196  val_loss=0.5036  val_LER=0.19201
  [LSTM] Epoch  40  tr_loss=0.4577  val_loss=0.4632  val_LER=0.18081
  [LSTM] Epoch  60  tr_loss=0.4338  val_loss=0.4379  val_LER=0.15508
  [LSTM] Epoch  80  tr_loss=0.4050  val_loss=0.4343  val_LER=0.14381
  [LSTM] Epoch 100  tr_loss=0.3984  val_loss=0.4330  val_LER=0.14401
  [LSTM] Early stop at epoch 104 (best=89)

  Test LER : 0.14607  (14.6067%)
  Trivial  : 0.20093  (20.0933%)
  vs Trivial : 1.4x


---
## 7. Save Models

In [8]:
os.makedirs("models", exist_ok=True)

for tag, model, ler, idim, hist in [
    ("multi_mlp_p1", mlp_p1,   ler_mlp_p1,  72,    hist_mlp_p1),
    ("multi_mlp_p2", mlp_p2,   ler_mlp_p2,  72,    hist_mlp_p2),
    ("multi_lstm",   lstm_mn,  ler_lstm_mn, STABS,  hist_lstm_mn),
]:
    ck = {
        'model_state': model.state_dict(),
        'test_ler':    ler,
        'history':     hist,
        'training':    'multi_noise',
    }
    if 'lstm' in tag:
        ck.update({'input_size': idim, 'lstm_hidden': 36,
                   'lstm_layers': 2, 'dense': [48,24,12]})
    else:
        ck.update({'input_dim': 72, 'hidden': [256,128,64]})
    torch.save(ck, f"models/decoder_{tag}.pt")
    print(f"  Saved models/decoder_{tag}.pt  (test_ler={ler:.5f})")

  Saved models/decoder_multi_mlp_p1.pt  (test_ler=0.15060)
  Saved models/decoder_multi_mlp_p2.pt  (test_ler=0.17567)
  Saved models/decoder_multi_lstm.pt  (test_ler=0.14607)


---
## 8. Quick Comparison: Multi-noise vs Single-noise

In [9]:
# Load single-noise results for comparison (from notebooks 02 and 05)
try:
    ck_sn_p1   = torch.load("models/decoder_part1.pt",    map_location='cpu', weights_only=False)
    ck_sn_lstm = torch.load("models/decoder_lstm.pt",     map_location='cpu', weights_only=False)
    sn_p1_ler  = ck_sn_p1['test_ler']
    sn_lstm_ler = ck_sn_lstm['test_ler']
    have_single = True
except FileNotFoundError:
    have_single = False

print("=" * 62)
print("Multi-noise vs Single-noise (mixed test set, all p levels)")
print("=" * 62)
print(f"  {'Model':<28}  {'Test LER':>10}  {'vs Trivial':>10}")
print("  " + "-"*52)
print(f"  {'Trivial':<28}  {trivial_ler:>10.4%}  {'—':>10}")
print(f"  {'MLP P1 (multi-noise)':<28}  {ler_mlp_p1:>10.4%}  {trivial_ler/ler_mlp_p1:>9.1f}x")
print(f"  {'MLP P2 (multi-noise)':<28}  {ler_mlp_p2:>10.4%}  {trivial_ler/ler_mlp_p2:>9.1f}x")
print(f"  {'LSTM (multi-noise)':<28}  {ler_lstm_mn:>10.4%}  {trivial_ler/ler_lstm_mn:>9.1f}x")
if have_single:
    print()
    print("  (Single-noise test LERs at p=0.001 only — not directly comparable)")
    print(f"  {'MLP P1 (single p=0.001)':<28}  {sn_p1_ler:>10.4%}")
    print(f"  {'LSTM (single p=0.001)':<28}  {sn_lstm_ler:>10.4%}")
print()
print("Run notebook 03B to see the full noise sweep comparison.")

Multi-noise vs Single-noise (mixed test set, all p levels)
  Model                           Test LER  vs Trivial
  ----------------------------------------------------
  Trivial                         20.0933%           —
  MLP P1 (multi-noise)            15.0600%        1.3x
  MLP P2 (multi-noise)            17.5667%        1.1x
  LSTM (multi-noise)              14.6067%        1.4x

  (Single-noise test LERs at p=0.001 only — not directly comparable)
  MLP P1 (single p=0.001)          1.4467%
  LSTM (single p=0.001)            1.0867%

Run notebook 03B to see the full noise sweep comparison.
